# EduTrack AI - Model Training and Evaluation
### SDG 4: Quality Education Capstone

This notebook trains an ensemble classifier (Random Forest + Logistic Regression) to predict whether a student is **At Risk** of failing a specific subject based on early-term features (midterm scores, attendance, study hours, and past history).

## 1. Load and Explore the Data

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the generated student dataset
data_path = "../data/student_data.csv"
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Check class distribution
risk_counts = df['At_Risk'].value_counts()
print("Risk Class Distribution:")
print(risk_counts)
print(f"Risk Rate: {df['At_Risk'].mean() * 100:.2f}%")

# Subject-wise risk distribution
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Subject', hue='At_Risk')
plt.title('At-Risk Students by Subject')
plt.xlabel('Subject')
plt.ylabel('Count')
plt.legend(title='At Risk', labels=['No', 'Yes'])
plt.show()

## 2. Feature-Target Setup and Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Define features and target variable
X = df[['Subject', 'Attendance_Rate', 'Study_Hours', 'Midterm_Score', 'Past_Score']]
y = df['At_Risk']

# Perform stratified split to maintain class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train records: {len(X_train)}")
print(f"Test records: {len(X_test)}")

## 3. Pipeline Construction (Preprocessing & Models)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression

# Define numerical and categorical columns
numeric_features = ['Attendance_Rate', 'Study_Hours', 'Midterm_Score', 'Past_Score']
categorical_features = ['Subject']

# Preprocessor setup
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# Individual classifiers
rf_clf = RandomForestClassifier(
    n_estimators=120, max_depth=6, random_state=42, class_weight='balanced'
)
lr_clf = LogisticRegression(
    random_state=42, class_weight='balanced', max_iter=1000
)

# Ensemble classifier using Soft Voting (averaging predicted probabilities)
ensemble = VotingClassifier(
    estimators=[('rf', rf_clf), ('lr', lr_clf)],
    voting='soft',
    weights=[1.2, 0.8]  # Weighted higher on Random Forest's nonlinear decision boundaries
)

# Combined Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ensemble)
])

## 4. Train Model Pipeline

In [ ]:
print("Training the pipeline...")
pipeline.fit(X_train, y_train)

## 5. Evaluate the Ensemble Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix, roc_curve

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Ensemble (AUC = {roc_auc_score(y_test, y_proba):.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

## 6. Save Model Pipeline

In [ ]:
# Save the pipeline model to the models folder
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "ensemble_model.pkl")

with open(model_path, "wb") as f:
    pickle.dump(pipeline, f)
    
print(f"Ensemble model pipeline successfully saved to: {model_path}")